# `holspec` quickstart

This notebook demonstrates a small end-to-end workflow: generate noisy triangular lattice point clouds, run the `holspec` pipeline, analyze the resulting spectra, and create a few representative visualizations.

## 1. Setup

Import packages, configure paths, and create output directories for the quickstart workflow.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml

%reload_ext autoreload
%autoreload 2

from holspec.paths import get_project_root
from holspec.point_data import PointDataEnsemble, run_data_generation
from holspec.pipeline import load_spectra, run_pipeline, trace_provenance

PROJECT_ROOT = get_project_root(Path.cwd())
EXAMPLES_DIR = PROJECT_ROOT / "examples"
CONFIGS_DIR = EXAMPLES_DIR / "configs"
DATA_DIR = EXAMPLES_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
INTERIM_DATA_DIR = DATA_DIR / "interim"
OUTPUTS_DIR = EXAMPLES_DIR / "outputs"

DATA_GENERATION_CONFIG_PATH = CONFIGS_DIR / "data_generation_quickstart.yml"
PIPELINE_CONFIG_PATH = CONFIGS_DIR / "pipeline_quickstart.yml"

for path in (CONFIGS_DIR, RAW_DATA_DIR, INTERIM_DATA_DIR, OUTPUTS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print("\nQuickstart directories:")
print(f"  Configs:      {CONFIGS_DIR.relative_to(PROJECT_ROOT)}")
print(f"  Data:         {DATA_DIR.relative_to(PROJECT_ROOT)}")
print(f"  Outputs:      {OUTPUTS_DIR.relative_to(PROJECT_ROOT)}")

Project root: /Users/USERNAME/Repositories/Work/holspec

Quickstart directories:
  Configs:      examples/configs
  Data:         examples/data
  Outputs:      examples/outputs


Define small helper functions used throughout the quickstart.

In [2]:
def format_noise_label(scale: float) -> str:
    return f"{scale:.2f}".replace(".", "p")


def make_dataset_label(n_rings: int, noise_scale: float) -> str:
    if np.isclose(noise_scale, 0.0):
        return f"triangular_lattice_nr{n_rings}_clean"
    return f"triangular_lattice_nr{n_rings}_noise_{format_noise_label(noise_scale)}"


def make_pipeline_config(
    input_filepaths: dict[str, str],
    simplicial_constructions: dict,
    metric_models: dict,
    spectra_settings: dict,
) -> dict:
    quiet_runtime = {"verbose": False}

    return {
        "summary": {"created_by": "examples/quickstart.ipynb"},
        "inputs": {
            "data_dir": str(RAW_DATA_DIR.relative_to(PROJECT_ROOT)),
            "filepaths": input_filepaths,
        },
        "stages": {
            "topology_simplicial": {
                "configs": {"simplicial_constructions": simplicial_constructions},
                "runtime": quiet_runtime | {
                    "cache_incidence": False,
                    "validate_boundary_property": False,
                },
            },
            "geometry_metric": {
                "configs": {"metric_models": metric_models},
                "runtime": quiet_runtime | {"validate_metric": True},
            },
            "hodge_laplacian": {
                "configs": {},
                "runtime": quiet_runtime | {
                    "cache_laplacians": False,
                    "validate_laplacians": False,
                },
            },
            "spectra": {
                "configs": spectra_settings,
                "runtime": quiet_runtime,
            },
        },
        "runtime": {"verbose": True, "save_stage_configs": False},
        "outputs": {"data_dir": str(INTERIM_DATA_DIR.relative_to(PROJECT_ROOT))},
    }

## 2. Configure the example

Define the triangular lattice noise sweep and write the data generation config.

In [3]:
# Define quickstart parameters.
CATEGORY = "quickstart"
N_RINGS = 5
NOISE_SCALES = [0.0, 0.10, 0.25, 0.40]
NUM_REALIZATIONS = 1
BASE_SEED = 42

# Build the data generation config.
data_generation_config = {
    "summary": {
        "created_by": "examples/quickstart.ipynb",
    },
    "configs": {
        CATEGORY: {},
    },
    "runtime": {
        "verbose": True,
    },
    "outputs": {
        "stage_name": "point_data",
        "data_dir": str(RAW_DATA_DIR.relative_to(PROJECT_ROOT)),
    },
}

# Generate one dataset per noise level.
for noise_scale in NOISE_SCALES:
    label = make_dataset_label(N_RINGS, noise_scale)
    data_generation_config["configs"][CATEGORY][label] = {
        "base_config": {
            "generator": "trilatthex",
            "params": {
                "n_rings": N_RINGS,
            },
        },
        "noise_config": {
            "scale": noise_scale,
            "distribution": "normal",
        },
        "num_realizations": NUM_REALIZATIONS,
        "base_seed": BASE_SEED,
    }

# Store labels for later quickstart steps.
DATASET_LABELS = list(data_generation_config["configs"][CATEGORY])
DATASET_NOISE_SCALES = {
    make_dataset_label(N_RINGS, noise_scale): noise_scale
    for noise_scale in NOISE_SCALES
}

# Write the config to YAML.
with open(DATA_GENERATION_CONFIG_PATH, "w") as f:
    yaml.safe_dump(data_generation_config, f, sort_keys=False)

print(f"Data generation config written to {DATA_GENERATION_CONFIG_PATH.relative_to(PROJECT_ROOT)}")
print("\nDatasets to generate:")
for label in DATASET_LABELS:
    print(f"  {label}")

Data generation config written to examples/configs/data_generation_quickstart.yml

Datasets to generate:
  triangular_lattice_nr5_clean
  triangular_lattice_nr5_noise_0p10
  triangular_lattice_nr5_noise_0p25
  triangular_lattice_nr5_noise_0p40


Define the pipeline settings and write the config used to run the quickstart workflow.

In [4]:
# Point the pipeline to generated point cloud files.
pipeline_input_filepaths = {
    label: str((RAW_DATA_DIR / CATEGORY / f"{label}.h5").relative_to(PROJECT_ROOT))
    for label in DATASET_LABELS
}

# Use Delaunay complexes for point cloud topology.
simplicial_constructions = {
    "delaunay": {
        "method": "delaunay",
        "params": {},
    },
}

# Use the combinatorial cochain metric.
metric_models = {
    "combinatorial": {
        "model": "combinatorial",
        "params": {},
    },
}

# Compute eigenvalue spectra with a dense solver.
spectra_settings = {
    "compute_spectra": True,
    "solver": "dense",
    "compute_eigenvectors": False,
}

# Build the pipeline config.
pipeline_config = make_pipeline_config(
    input_filepaths=pipeline_input_filepaths,
    simplicial_constructions=simplicial_constructions,
    metric_models=metric_models,
    spectra_settings=spectra_settings,
)

# Write the config to YAML.
with open(PIPELINE_CONFIG_PATH, "w") as f:
    yaml.safe_dump(pipeline_config, f, sort_keys=False)

print(f"Pipeline config written to {PIPELINE_CONFIG_PATH.relative_to(PROJECT_ROOT)}")
print("\nPipeline inputs:")
for label, filepath in pipeline_input_filepaths.items():
    print(f"  {label}: {filepath}")

Pipeline config written to examples/configs/pipeline_quickstart.yml

Pipeline inputs:
  triangular_lattice_nr5_clean: examples/data/raw/quickstart/triangular_lattice_nr5_clean.h5
  triangular_lattice_nr5_noise_0p10: examples/data/raw/quickstart/triangular_lattice_nr5_noise_0p10.h5
  triangular_lattice_nr5_noise_0p25: examples/data/raw/quickstart/triangular_lattice_nr5_noise_0p25.h5
  triangular_lattice_nr5_noise_0p40: examples/data/raw/quickstart/triangular_lattice_nr5_noise_0p40.h5


## 3. Generate point cloud data

Generate the point cloud ensembles from the quickstart data generation config.

In [5]:
# Data generation will go here.

## 4. Run the `holspec` pipeline

Build simplicial complexes, define cochain metrics, assemble Hodge Laplacians, and compute eigenvalue spectra from the quickstart pipeline config.

In [6]:
# Pipeline execution will go here.

## 5. Analyze spectra

Configure the spectral analysis and analyze the eigenvalue spectra from the pipeline run.

In [7]:
# Spectral analysis will go here.

## 6. Visualize outputs

Plot selected point clouds, simplicial complexes, and eigenvalue distributions to visualize how the spectra respond to structural change.

In [8]:
# Visualization will go here.